# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring.

The task is a scoring and ranking problem. I want to identify website content pages that may need review because their search performance is declining.

The system will assign a priority score to each page and help editors decide which pages to review first. The score will support human decisions rather than automatically changing content.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_name = "Refresh / Content Opportunity Scoring"
task_type = "Scoring / Ranking"

print("Lane:", lane_name)
print("Task type:", task_type)
print("Action: Prioritize content pages for editorial review")

Lane: Refresh / Content Opportunity Scoring
Task type: Scoring / Ranking
Action: Prioritize content pages for editorial review


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is whether a content page is declining.

I will use `trend_direction` to define a binary proxy label: 1 if the trend is "down" and 0 otherwise. This is a rule-derived label, not a direct measure of whether a page genuinely needs refreshing.

The label can help us investigate which page characteristics are associated with declining performance. It does not prove that refreshing a page will improve its performance.


**Limitation and leakage risk:** The target is derived from `trend_direction`, so it is a proxy rather than a verified refresh outcome. If I train a model to predict this label, I must exclude `trend_direction` and any equivalent information used to create the label from the input features. Otherwise, the model could access the answer indirectly.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target_name = "is_declining_label"
target_source = "trend_direction"

print("Target:", target_name)
print("Source column:", target_source)
print("Label definition: 1 = down, 0 = not down")

Target: is_declining_label
Source column: trend_direction
Label definition: 1 = down, 0 = not down


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is Precision@50.

It measures the proportion of the top 50 pages selected by the scoring system that are actually labeled as declining.

I will compare the learned ranking against a simple rule-based baseline. Higher Precision@50 means more declining pages are found near the top of the review queue. This evaluation measures agreement with the proxy label, not the causal impact of refreshing a page.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
primary_metric = "Precision@50"
evaluation_cutoff = 50

print("Primary metric:", primary_metric)
print("Evaluation cutoff:", evaluation_cutoff)
print("A valid evaluation needs a held-out dataset and a defined ranking score.")

Primary metric: Precision@50
Evaluation cutoff: 50
A valid evaluation needs a held-out dataset and a defined ranking score.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one website content page. Each row represents one page, and each column represents an observed feature or attribute, such as search performance or content freshness.

I will inspect the real dataset, check the available columns, and create the proxy target from `trend_direction`. I will show a small dataframe sample and check the target distribution.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import subprocess
import pandas as pd

# Find the dataset if the repository is already available.
candidates = [
    Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)

# If needed, clone your public repository into Colab.
if data_path is None:
    repo_path = Path("/content/flyrank-ml-internship")

    if not repo_path.exists():
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/HadiaWaheed/flyrank-ml-internship.git",
                str(repo_path),
            ],
            check=True,
        )

    data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Check the repository and CSV path."
    )

df = pd.read_csv(data_path)

print("Dataset path:", data_path)
print("Rows and columns:", df.shape)
print("Available columns:", df.columns.tolist())

assert "trend_direction" in df.columns, (
    "Expected column 'trend_direction' was not found."
)

# Create the proxy target.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
    .astype("int8")
)

# Display a real dataframe.
display(df.head(10))

print("Total pages:", len(df))
print("\nTarget distribution:")
display(
    df["is_declining_label"]
    .value_counts()
    .rename(index={0: "Not down", 1: "Down"})
    .to_frame("page_count")
)

print(
    "Declining label percentage:",
    round(df["is_declining_label"].mean() * 100, 1),
    "%"
)

Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Rows and columns: (30000, 44)
Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9,1
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2,1


Total pages: 30000

Target distribution:


,page_count
is_declining_label,
Down,16262
Not down,13738


Declining label percentage: 54.2 %


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could prioritize pages using only one signal, such as low traffic or an old update date. However, declining performance may be associated with several signals working together.

A machine learning model can learn combinations of signals from historical examples and assign scores to pages. I will compare this approach with a transparent rule-based baseline instead of assuming ML is automatically better.

The output will help editors decide which pages to inspect. It cannot prove that a content refresh will cause better search performance or predict Google's ranking algorithm.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fixed_rule_example = (
    "Prioritize pages using one predefined condition, "
    "such as a decline in observed performance."
)

ml_approach = (
    "Learn patterns from multiple page features and "
    "compare the resulting ranking with the fixed rule."
)

print("Fixed-rule baseline:", fixed_rule_example)
print("ML approach:", ml_approach)
print("Comparison metric:", primary_metric)

Fixed-rule baseline: Prioritize pages using one predefined condition, such as a decline in observed performance.
ML approach: Learn patterns from multiple page features and compare the resulting ranking with the fixed rule.
Comparison metric: Precision@50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.
- [ ] I documented the proxy target's limitations and checked for target leakage.
- [ ] I have not claimed a measured Precision@50 result without evaluating a ranking.
 Done.